In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.signal import remez, freqz
from ipywidgets import Dropdown, FloatSlider, VBox, HBox, HTML, Layout
from IPython.display import display

# ============================================================
# FIR HILBERT TRANSFORMER
# ============================================================

plt.ioff()
CONTENT_WIDTH = '1000px'

plt.rcParams.update({'font.size':11,'axes.titlesize':12.5,'axes.labelsize':11,'xtick.labelsize':9.5,'ytick.labelsize':9.5,'legend.fontsize':8.8})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>
.ht-root{width:1000px;max-width:1000px;font-family:Arial,sans-serif;}
.ht-header{background:linear-gradient(90deg,#00695c,#00897b);color:white;padding:11px 15px;border-radius:8px 8px 0 0;font-size:18px;font-weight:bold;}
.ht-doc{background:#f5fbfa;border:1px solid #b8d8d2;border-top:none;padding:11px 14px;border-radius:0 0 8px 8px;font-size:14px;line-height:1.55;margin-bottom:9px;}
.ht-box{width:100%;box-sizing:border-box;border:1px solid #b8d8d2;border-radius:7px;padding:9px 11px;margin-bottom:8px;font-size:13.5px;line-height:1.48;}
.ht-title{font-weight:bold;color:#00695c;font-size:14.5px;margin-bottom:6px;}
.ht-cols{display:flex;gap:14px;align-items:flex-start;flex-wrap:nowrap;}
.ht-col{flex:1;min-width:0;}
.widget-label{font-size:13px !important;}
.jupyter-widgets input,.jupyter-widgets select{font-size:13px !important;}
.jupyter-widgets-output-area,.widget-output,.output_area,.output_subarea,.jp-OutputArea-output,.jp-OutputArea-child{overflow-x:visible !important;overflow-y:visible !important;max-width:none !important;}
.jp-OutputArea,.output_wrapper,.widget-box,.jupyter-widgets{overflow:visible !important;}
</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="ht-root">
<div class="ht-header">FIR Hilbert Transformer</div>
<div class="ht-doc">
<b>Purpose.</b>
An ideal Hilbert transformer has unit magnitude and introduces a phase shift of ±90°.
A practical FIR implementation operates only over the frequency range occupied by the input signal.
<br><br>
<b>Filter structure.</b>
The impulse response is antisymmetric. An odd FIR length produces a Type-III filter, which has forced zeros at both ω=0 and ω=π.
An even FIR length produces a Type-IV filter, which has a forced zero at ω=0 but not at ω=π.
<br><br>
<b>Parks–McClellan design.</b>
Within the specified useful bandwidth the desired magnitude is unity and the weight is constant.
The filter is designed with <code>scipy.signal.remez(..., type='hilbert')</code>.
</div>
</div>
"""))

# ============================================================
# CONTROLS
# ============================================================

filter_type = Dropdown(options=['Type III','Type IV'],value='Type III',description='FIR type:',style={'description_width':'70px'},layout=Layout(width='220px'))
length_control = Dropdown(options=list(range(11,52,2)),value=21,description='Length N:',style={'description_width':'70px'},layout=Layout(width='220px'))
f1_slider = FloatSlider(value=0.10,min=0.02,max=0.40,step=0.01,description='ω1 / π:',continuous_update=True,readout_format='.2f',style={'description_width':'70px'},layout=Layout(width='280px'))
f2_slider = FloatSlider(value=0.90,min=0.60,max=0.98,step=0.01,description='ω2 / π:',continuous_update=True,readout_format='.2f',style={'description_width':'70px'},layout=Layout(width='280px'))

controls = VBox([
    HTML('<div class="ht-title">Hilbert-transformer parameters</div>'),
    HBox([filter_type,length_control,f1_slider,f2_slider])
],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b8d8d2',padding='9px 12px',margin='0 0 8px 0'))

info = HTML(layout=Layout(width=CONTENT_WIDTH,margin='0 0 8px 0'))

# ============================================================
# LENGTH OPTIONS
# ============================================================

def update_length_options(change=None):
    if filter_type.value == 'Type III':
        options = list(range(11,52,2))
        default = 21
    else:
        options = list(range(12,53,2))
        default = 20

    old_value = length_control.value
    length_control.options = options

    if old_value in options:
        length_control.value = old_value
    else:
        length_control.value = default

# ============================================================
# FIGURE — CREATED ONCE
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(10.0,7.0))
ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# MAGNITUDE
# ============================================================

line_mag, = ax1.plot([],[],color='red',linewidth=1.5,label='Hilbert transformer')
desired_mag, = ax1.plot([],[],'--',linewidth=1.0,label='Desired magnitude')
left_edge = ax1.axvline(0.10,linestyle=':',linewidth=1.0)
right_edge = ax1.axvline(0.90,linestyle=':',linewidth=1.0)

ax1.set_xlim(0,1)
ax1.set_ylim(0,1.15)
ax1.set_title('Hilbert Transformer Magnitude Response')
ax1.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax1.set_ylabel(r'$|H(e^{j\omega})|$')
ax1.grid(True,linestyle=':',alpha=0.25)
ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.18),ncol=2,frameon=False)

# ============================================================
# MAGNITUDE ERROR
# ============================================================

line_error, = ax2.plot([],[],color='red',linewidth=1.5,label='Magnitude error')
zero_error = ax2.axhline(0,linestyle='--',linewidth=1.0,label='Zero error')

ax2.set_title('Passband Magnitude Error')
ax2.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax2.set_ylabel(r'$1-|H(e^{j\omega})|$')
ax2.grid(True,linestyle=':',alpha=0.25)
ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.18),frameon=False)

# ============================================================
# PHASE AFTER REMOVING LINEAR DELAY
# ============================================================

line_phase, = ax3.plot([],[],color='red',linewidth=1.5,label='Phase after delay removal')
ideal_phase = ax3.axhline(90,linestyle='--',linewidth=1.0,label=r'Ideal $+90^\circ$')

ax3.set_ylim(60,120)
ax3.set_title('Hilbert Phase Shift')
ax3.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax3.set_ylabel('Phase (degrees)')
ax3.grid(True,linestyle=':',alpha=0.25)
ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.18),frameon=False)

# ============================================================
# IMPULSE RESPONSE
# ============================================================

impulse_markers, = ax4.plot([],[],'ro',markersize=3.4)
stem_collection = None
center_line = ax4.axvline(0,linestyle='--',linewidth=1.0,label='Antisymmetry center')

ax4.set_title('Antisymmetric FIR Impulse Response')
ax4.set_xlabel('Sample index $n$')
ax4.set_ylabel('$h[n]$')
ax4.grid(True,linestyle=':',alpha=0.25)
ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.18),frameon=False)

plt.subplots_adjust(left=0.075,right=0.985,top=0.95,bottom=0.14,wspace=0.28,hspace=0.65)

# ============================================================
# UPDATE
# ============================================================

def update_hilbert(change=None):
    global stem_collection

    N = length_control.value
    f1 = f1_slider.value
    f2 = f2_slider.value
    ftype = filter_type.value

    if f2 <= f1:
        info.value = '<div class="ht-root"><div class="ht-box"><b>Invalid bandwidth:</b> ω2 must be greater than ω1.</div></div>'
        return

    try:
        h = remez(N,[f1,f2],[1.0],type='hilbert',fs=2.0,maxiter=100,grid_density=32)

    except Exception as e:
        info.value = f'<div class="ht-root"><div class="ht-box"><b>Design error:</b> {e}</div></div>'
        return

    omega,H = freqz(h,worN=32768)
    fn = omega/np.pi
    mag = np.abs(H)

    mask = (fn >= f1) & (fn <= f2)
    fn_band = fn[mask]
    mag_band = mag[mask]

    error = 1.0-mag_band

    delay = (N-1)/2
    H_corrected = H*np.exp(1j*omega*delay)
    phase_corrected = np.unwrap(np.angle(H_corrected))
    phase_deg = np.rad2deg(phase_corrected)

    phase_band = phase_deg[mask]
    phase_band = ((phase_band+180)%360)-180

    symmetry_error = np.max(np.abs(h+h[::-1]))
    H0 = abs(np.sum(h))
    Hpi = abs(np.sum(h*(-1)**np.arange(N)))

    info.value = f"""
    <div class="ht-root">
    <div class="ht-box">
    <div class="ht-title">Current Hilbert-transformer design</div>

    <div class="ht-cols">

    <div class="ht-col">
    Structure: <b>{ftype}</b><br>
    FIR length: <b>N = {N}</b><br>
    Filter order: <b>{N-1}</b>
    </div>

    <div class="ht-col">
    Useful bandwidth: <b>{f1:.2f}π ≤ ω ≤ {f2:.2f}π</b><br>
    Desired magnitude: <b>|H<sub>d</sub>| = 1</b>
    </div>

    <div class="ht-col">
    Antisymmetry error: <b>{symmetry_error:.2e}</b><br>
    |H(0)|: <b>{H0:.2e}</b><br>
    |H(π)|: <b>{Hpi:.2e}</b>
    </div>

    </div>
    </div>
    </div>
    """

    # ========================================================
    # MAGNITUDE
    # ========================================================

    line_mag.set_data(fn,mag)
    desired_mag.set_data([f1,f2],[1,1])
    left_edge.set_xdata([f1,f1])
    right_edge.set_xdata([f2,f2])

    # ========================================================
    # MAGNITUDE ERROR
    # ========================================================

    line_error.set_data(fn_band,error)

    ax2.set_xlim(f1,f2)

    emax = max(np.max(np.abs(error)),1e-4)
    ax2.set_ylim(-1.15*emax,1.15*emax)

    # ========================================================
    # PHASE
    # ========================================================

    line_phase.set_data(fn_band,phase_band)

    ax3.set_xlim(f1,f2)

    # ========================================================
    # IMPULSE RESPONSE
    # ========================================================

    n = np.arange(N)
    impulse_markers.set_data(n,h)

    if stem_collection is not None:
        stem_collection.remove()

    segments = [np.array([[ni,0],[ni,hi]]) for ni,hi in zip(n,h)]
    stem_collection = LineCollection(segments)
    ax4.add_collection(stem_collection)

    center = (N-1)/2
    center_line.set_xdata([center,center])
    center_line.set_label(f'Antisymmetry center = {center:g}')

    ax4.set_xlim(-1,N)

    hmax = max(np.max(np.abs(h)),0.05)
    ax4.set_ylim(-1.15*hmax,1.15*hmax)

    fig.canvas.draw_idle()

# ============================================================
# EVENTS
# ============================================================

def type_changed(change):
    update_length_options()
    update_hilbert()

filter_type.observe(type_changed,names='value')
length_control.observe(update_hilbert,names='value')
f1_slider.observe(update_hilbert,names='value')
f2_slider.observe(update_hilbert,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(controls)
display(info)
display(fig.canvas)

update_hilbert()